# 03. Huấn luyện & Ablation Study cho PhoBERT (Kaggle / Colab)

Notebook của **Đức Anh**: thiết lập môi trường trên Kaggle/Colab, kiểm tra BPE subword alignment, huấn luyện `PhoBERT` (`vinai/phobert-base-v2`) thông qua `scripts/train.py`, chạy các nhánh Ablation (`freeze_backbone`, `concat_last_4`, `use_crf=true`) và phân tích định tính.

## Setup Environment (Kaggle / Colab / W&B)

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/TTins1702/NER-PhoNER_COVID19-UET.git"
BRANCH = "tin-bilstm-crf"
REPO_DIR = "NER-PhoNER_COVID19-UET"
if Path("../src").exists():
    %cd ..
if not Path("src").exists():
    if not Path(REPO_DIR).exists():
        !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_DIR}
else:
    !git pull

!pip install -q -r requirements.txt
!python -m src.data.download --output-dir data

if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if os.environ.get("WANDB_API_KEY"):
    os.environ.pop("WANDB_MODE", None)
    print("WANDB_API_KEY loaded -> logging online to Weights & Biases")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_API_KEY not set -> running in WANDB_MODE=offline")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
from src.utils.config import load_config

cfg = load_config("configs/phobert.yaml")
print("Setup done. Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Week 1: BPE Tokenizer Check

In [ ]:
from transformers import AutoTokenizer
from src.data.phobert_dataset import inspect_bpe_sample

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["pretrained_name"])
sample_words = ["Bệnh_nhân", "ở", "Vĩnh_Phúc", "bị", "sốt", "cao", "."]
for word, subwords in inspect_bpe_sample(tokenizer, sample_words):
    print(f"{word} -> {subwords}")

## Week 2: Dataset, Alignment Check & Sanity Batch

In [ ]:
from functools import partial
from torch.utils.data import DataLoader
from src.data.download import load_phoner_split
from src.data.phobert_dataset import PhoBERTDataset, phobert_collate_fn

data_dir = cfg["data"]["data_dir"]
level = cfg["data"]["level"]
max_seq_length = int(cfg["data"]["max_seq_length"])

datasets = {}
for split in ("train", "dev", "test"):
    records = load_phoner_split(data_dir, level=level, split=split)
    ds = PhoBERTDataset(records, tokenizer=tokenizer, max_seq_length=max_seq_length)
    datasets[split] = ds
    print(split, ds.get_truncation_stats())

first_item = datasets["train"][0]
print("Alignment match on sample 0:", int(first_item["word_mask"].sum().item()) == first_item["orig_length"])

collate_fn = partial(phobert_collate_fn, pad_token_id=tokenizer.pad_token_id)
sample_loader = DataLoader(datasets["train"], batch_size=cfg["training"]["batch_size"], shuffle=False, collate_fn=collate_fn)
sample_batch = next(iter(sample_loader))
print("Batch tensor shapes:", {k: tuple(v.shape) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)})

# (Tuỳ chọn) Chạy sanity check 1 epoch:
# !python scripts/train.py --config configs/phobert.yaml --override experiment_name=phobert_week2_sanity training.epochs=1 training.checkpoint_dir=checkpoints/phobert/phobert_week2_sanity

## Week 3: Main PhoBERT Fine-tuning Run

In [ ]:
# 1. Mô hình chuẩn: PhoBERT Full Fine-tune + Last Layer + Softmax
!python scripts/train.py --config configs/phobert.yaml

import json

main_json_path = Path(cfg["logging"]["output_dir"]) / f"{cfg['experiment_name']}.json"
main_result = json.loads(main_json_path.read_text(encoding="utf-8"))
t = main_result["test"]
print(
    f"Test overall | P={t['micro_precision']:.4f} | R={t['micro_recall']:.4f} | "
    f"micro-F1={t['micro_f1']:.4f} | macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
)
for label, scores in t["per_entity"].items():
    print(
        f"{label:22s} P={scores['precision']:.4f} R={scores['recall']:.4f} "
        f"F1={scores['f1']:.4f} (n={scores['support']})"
    )

## Week 4: PhoBERT Ablation Study

In [ ]:
# 2. Ablation 1: Đóng băng backbone PhoBERT (chỉ train classifier head)
!python scripts/train.py --config configs/phobert.yaml --override experiment_name=phobert_freeze_last_softmax model.freeze_backbone=true training.checkpoint_dir=checkpoints/phobert/phobert_freeze_last_softmax

# 3. Ablation 2: Nối 4 tầng ẩn cuối cùng của PhoBERT
!python scripts/train.py --config configs/phobert.yaml --override experiment_name=phobert_full_concat4_softmax model.feature_mode=concat_last_4 training.checkpoint_dir=checkpoints/phobert/phobert_full_concat4_softmax

# 4. Ablation 3: PhoBERT + LinearChainCRF head
!python scripts/train.py --config configs/phobert.yaml --override experiment_name=phobert_full_last_crf model.use_crf=true training.checkpoint_dir=checkpoints/phobert/phobert_full_last_crf

metrics_dir = Path(cfg["logging"]["output_dir"])
for path in sorted(metrics_dir.glob("phobert*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    t = data["test"]
    print(
        f"{data['experiment_name']:30s} | P={t['micro_precision']:.4f} | "
        f"R={t['micro_recall']:.4f} | micro-F1={t['micro_f1']:.4f} | "
        f"macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
    )

## Week 5: Qualitative Analysis (Long Entities & BiLSTM-CRF Comparison)

In [ ]:
from src.data.phobert_dataset import decode_word_level_predictions
from src.training.trainer import build_model_and_dataloaders

metrics_dir = Path(cfg["logging"]["output_dir"])
phobert_json_path = metrics_dir / f"{cfg['experiment_name']}.json"
bilstm_json_path = metrics_dir / "bilstm_crf_word.json"

if phobert_json_path.exists():
    phobert_res = json.loads(phobert_json_path.read_text(encoding="utf-8"))
    for entity_name in ["SYMPTOM_AND_DISEASE", "LOCATION"]:
        if entity_name in phobert_res["test"]["per_entity"]:
            print(entity_name, "F1:", round(phobert_res["test"]["per_entity"][entity_name]["f1"], 4))
    print("PhoBERT test micro_f1:", round(phobert_res["test"]["micro_f1"], 4))
    if bilstm_json_path.exists():
        bilstm_res = json.loads(bilstm_json_path.read_text(encoding="utf-8"))
        print("BiLSTM-CRF test micro_f1:", round(bilstm_res["test"]["micro_f1"], 4))

ckpt_path = Path(cfg["training"]["checkpoint_dir"]) / "best_model.pt"
if ckpt_path.exists():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    ckpt_cfg = ckpt["config"]
    test_records = load_phoner_split(ckpt_cfg["data"]["data_dir"], level=ckpt_cfg["data"]["level"], split="test")
    model, loaders, _ = build_model_and_dataloaders(ckpt_cfg, {"test": test_records})
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    shown = 0
    with torch.no_grad():
        for batch in loaders["test"]:
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                word_mask=batch["word_mask"].to(device),
            )
            preds = decode_word_level_predictions(out["predictions"], batch["orig_lengths"])
            for words, gold, pred in zip(batch["words"], batch["gold_tags"], preds):
                span_len = 0
                has_long_entity = False
                for tag in gold:
                    if tag.startswith("B-"):
                        span_len = 1
                    elif tag.startswith("I-") and span_len > 0:
                        span_len += 1
                    else:
                        span_len = 0
                    if span_len >= 3:
                        has_long_entity = True
                        break
                if has_long_entity:
                    pairs = [(w, g, p) for w, g, p in zip(words, gold, pred) if g != "O" or p != "O"]
                    print("Long-entity example:", pairs)
                    shown += 1
                    if shown >= 5:
                        break
            if shown >= 5:
                break
else:
    print("Run Week 3 first to generate", ckpt_path)

## Week 6: Reproducibility & Output Summary

In [ ]:
metrics_dir = Path(cfg["logging"]["output_dir"])
if metrics_dir.exists():
    for path in sorted(metrics_dir.glob("*.json")):
        data = json.loads(path.read_text(encoding="utf-8"))
        t = data.get("test", {})
        print(
            f"{data.get('experiment_name', path.stem):30s} | "
            f"P={t.get('micro_precision', 0.0):.4f} | "
            f"R={t.get('micro_recall', 0.0):.4f} | "
            f"micro-F1={t.get('micro_f1', 0.0):.4f} | "
            f"macro-F1={t.get('macro_f1', 0.0):.4f} | "
            f"BIO_viol={t.get('bio_violations', 0)}"
        )